In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import torch
import numpy as np
import yfinance as yf
from operator import itemgetter
from models.lstm_model import LSTM_Model
from utils import get_device, build_seq, plus_bus_day

# Parameter

In [3]:
ckpt_path = '../checkpoints/cnn_checkpoint.pt'

ticker = 'VOO'
predict_from = '2026-01-01'
predict_to   = '2026-03-31'

# Extract saved objects and configs

In [4]:
device = get_device()
checkpoint = torch.load("../checkpoints/lstm_checkpoint.pt", map_location = device)

In [5]:
model_config = checkpoint["model_config"]
x_scaler = checkpoint["x_scaler"]
y_scaler = checkpoint["y_scaler"]
input_cols = checkpoint["input_cols"]
seq_len = checkpoint["preprocess_config"]["seq_length"]
steps_ahead = checkpoint["steps_ahead"]
target_cols = checkpoint['target_cols']

# Rebuild model

In [6]:
model = LSTM_Model(**model_config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
_ = model.eval()

# Get Latest Data

In [7]:
df = yf.download(ticker, period="6mo")
data = df.dropna().copy()

data = data.reset_index()
data.columns = ['Date', 'Close', 'High', 'Low', "Open", "Volume"]


[*********************100%***********************]  1 of 1 completed


# Prepare Data

In [8]:
X_latest = x_scaler.transform(data[input_cols].values) # .astype(np.float32)
y_latest = y_scaler.fit_transform(data[target_cols].values)

In [9]:
Date_all = data['Date'].values.flatten().tolist()
Seqs = build_seq( seq_len, X_latest, steps_ahead, Date_all, y_latest, True)
X_seq, X_Dates, y_Seq = itemgetter('X_Seq','X_id','y_Seq')(Seqs)

In [11]:
for id, item in enumerate(X_Dates):
    
    fr = item[0].strftime('%Y-%m-%d')
    to = item[-1].strftime('%Y-%m-%d')

    actual = y_scaler.inverse_transform([y_Seq[id]])[0, 0] if y_Seq[id] is not None else None

    
    predict_day = plus_bus_day(to, steps_ahead)

    predict_seq = X_seq[id]
    Predict_Tensor = torch.tensor(predict_seq, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        
        pred_scaled = model(Predict_Tensor).cpu().numpy()    
        pred_real = y_scaler.inverse_transform(pred_scaled)[0, 0]
        
        
    print(f"Use Data From {fr} to {to} to predict {predict_day}" )
    
    if actual:
        
        print(f"The Actual Predicted price is ${actual:.2f}")
        
    print(f"The Predicted price is ${pred_real:.2f} \n")

Use Data From 2025-10-29 to 2025-11-11 to predict 2025-11-18
The Actual Predicted price is $603.38
The Predicted price is $651.98 

Use Data From 2025-10-30 to 2025-11-12 to predict 2025-11-19
The Actual Predicted price is $605.57
The Predicted price is $651.99 

Use Data From 2025-10-31 to 2025-11-13 to predict 2025-11-20
The Actual Predicted price is $596.38
The Predicted price is $651.97 

Use Data From 2025-11-03 to 2025-11-14 to predict 2025-11-21
The Actual Predicted price is $602.32
The Predicted price is $651.87 

Use Data From 2025-11-04 to 2025-11-17 to predict 2025-11-24
The Actual Predicted price is $611.28
The Predicted price is $651.71 

Use Data From 2025-11-05 to 2025-11-18 to predict 2025-11-25
The Actual Predicted price is $616.96
The Predicted price is $651.48 

Use Data From 2025-11-06 to 2025-11-19 to predict 2025-11-26
The Actual Predicted price is $621.23
The Predicted price is $651.22 

Use Data From 2025-11-07 to 2025-11-20 to predict 2025-11-28
The Actual Pred

In [ ]:
# for item in x_scaler.inverse_transform( X_tensor.reshape(-1,len(input_cols))):
#     # print(item)

#     for x in item:
        
#         print(f"{x:.6f}" )   